# SFT Qwen2.5-0.5B bằng LoRA trên bộ instruction tiếng Việt

Bài 01 đã làm sạch `OpenOrca-Viet` và ghi `m14-train.jsonl`, `m14-val.jsonl`. Ở đây bạn tự viết vòng SFT (không dùng
`Trainer`): dữ liệu qua `tokenize_chat` và `pad_batch` của bài Basic 02, batch theo **ngân sách token** để bớt padding,
LoRA của PEFT trên mọi lớp tuyến tính, rồi đo **trước và sau** bằng hai thước đo không cần LLM chấm: loss (và perplexity)
trên phần trả lời của tập validation chia theo nhóm, và exact-match trên các câu hỏi trắc nghiệm có đáp án ngắn. Cuối
bài so với một model lớn hơn **không fine-tune**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) tải model, train, đo; test bỏ qua chúng.
- Cần `data/raw/openorca-viet/cache/m14-train.jsonl` và `m14-val.jsonl` do bài 01 ghi ra: làm bài 01 trước.
- Chạy thật bằng kernel `ai-gpu` (GPU 4GB là đủ). Kernel CPU chỉ chạy bản thu nhỏ để kiểm code chạy hết.
- Chấm: `uv run pytest tracks/ai-engineer/modules/14-fine-tuning-llm/exercises/intermediate/02-sft-instruction -q`

In [ ]:
# explore: tìm gốc repo, kiểm tệp của bài 01
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


CACHE = dataset_dir("openorca-viet") / "cache"
for name in ("m14-train.jsonl", "m14-val.jsonl"):
    if not (CACHE / name).exists():
        raise RuntimeError(f"Chưa có {name}: chạy hết notebook bài 01 trước.")
print(CACHE)

In [ ]:
import math
import re
import unicodedata

import numpy as np
import torch

IGNORE_INDEX = -100

## 1. Dữ liệu: dán từ Basic 02, batch theo ngân sách token

- **Dán** `tokenize_chat` và `pad_batch` của bạn ở bài Basic 02 (giữ nguyên chữ ký).
- `token_budget_batches(lengths, max_tokens, seed=0) -> list[list[int]]`: xếp chỉ số mẫu theo độ dài tăng dần (sort ổn
  định), rồi lần lượt gom vào batch hiện tại khi `số mẫu × độ dài lớn nhất` của batch sau khi thêm vẫn không vượt
  `max_tokens`; vượt thì mở batch mới (một mẫu dài hơn `max_tokens` đứng một mình). Cuối cùng trộn **thứ tự các
  batch** bằng `np.random.default_rng(seed).permutation`. Mỗi chỉ số xuất hiện đúng một lần.

In [ ]:
def tokenize_chat(messages: list[dict], tokenizer, max_length: int | None = None) -> dict | None:
    """Dán từ bài Basic 02."""
    def ids_of(text):
        return list(tokenizer(text, add_special_tokens=False)["input_ids"])

    ids = ids_of(tokenizer.apply_chat_template(messages, tokenize=False))
    labels = [IGNORE_INDEX] * len(ids)
    for i, message in enumerate(messages):
        if message["role"] != "assistant":
            continue
        prompt = ids_of(tokenizer.apply_chat_template(messages[:i], tokenize=False, add_generation_prompt=True))
        upto = ids_of(tokenizer.apply_chat_template(messages[: i + 1], tokenize=False))
        if ids[: len(prompt)] != prompt or ids[: len(upto)] != upto:
            raise ValueError("chat template không cho tiền tố ổn định")
        labels[len(prompt): len(upto)] = ids[len(prompt): len(upto)]
    if max_length is not None:
        ids, labels = ids[:max_length], labels[:max_length]
    if all(label == IGNORE_INDEX for label in labels):
        return None
    return {"input_ids": ids, "labels": labels}


def pad_batch(examples: list[dict], pad_token_id: int, multiple_of: int = 1) -> dict[str, torch.Tensor]:
    """Dán từ bài Basic 02."""
    longest = max(len(e["input_ids"]) for e in examples)
    width = -(-longest // multiple_of) * multiple_of
    input_ids = torch.full((len(examples), width), pad_token_id, dtype=torch.long)
    labels = torch.full((len(examples), width), IGNORE_INDEX, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), width), dtype=torch.long)
    for row, e in enumerate(examples):
        n = len(e["input_ids"])
        input_ids[row, :n] = torch.tensor(e["input_ids"])
        labels[row, :n] = torch.tensor(e["labels"])
        attention_mask[row, :n] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def token_budget_batches(lengths: list[int], max_tokens: int, seed: int = 0) -> list[list[int]]:
    order = sorted(range(len(lengths)), key=lambda i: lengths[i])
    batches, current, longest = [], [], 0
    for i in order:
        new_longest = max(longest, lengths[i])
        if current and new_longest * (len(current) + 1) > max_tokens:
            batches.append(current)
            current, new_longest = [], lengths[i]
        current.append(i)
        longest = new_longest
    if current:
        batches.append(current)
    perm = np.random.default_rng(seed).permutation(len(batches))
    return [batches[k] for k in perm]

## 2. Đo: loss trên phần trả lời, exact-match câu trắc nghiệm

- `heldout_nll(model, batches) -> tuple[float, int]`: `batches` là các dict của `pad_batch`. Chạy model ở chế độ
  `eval`, không tính gradient, cộng **tổng** negative log-likelihood của mọi token có label (dự đoán token `t` từ logits
  ở vị trí `t - 1`) trên mọi batch, chia cho tổng số token đó. Trả `(nll trung bình mỗi token, số token)`; perplexity là
  `exp` của nó. Đừng lấy trung bình của các trung bình batch: batch ngắn sẽ nặng ký bằng batch dài.
- `normalize_answer(text) -> str`: NFC, chữ thường, bỏ mọi ký tự không phải chữ/số/khoảng trắng, gộp khoảng trắng, strip.
- `is_short_answer(messages, max_chars=60) -> bool`: câu trả lời (assistant) sau `normalize_answer` khác rỗng, dài
  không quá `max_chars` ký tự, và xuất hiện nguyên văn trong câu hỏi (user) đã `normalize_answer`: kiểu câu trắc nghiệm
  "Tùy chọn: - ... - ..." mà đáp án là một lựa chọn.
- `exact_match(prediction, gold) -> bool`: bằng nhau sau `normalize_answer`.

In [ ]:
@torch.no_grad()
def heldout_nll(model, batches: list[dict[str, torch.Tensor]]) -> tuple[float, int]:
    model.eval()
    device = next(model.parameters()).device
    total, count = 0.0, 0
    for batch in batches:
        logits = model(input_ids=batch["input_ids"].to(device),
                       attention_mask=batch["attention_mask"].to(device)).logits[:, :-1].float()
        labels = batch["labels"][:, 1:].to(device)
        keep = labels != IGNORE_INDEX
        total += torch.nn.functional.cross_entropy(logits[keep], labels[keep], reduction="sum").item()
        count += int(keep.sum())
    return total / count, count


def normalize_answer(text: str) -> str:
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\s]|_", " ", text)
    return " ".join(text.split())


def is_short_answer(messages: list[dict], max_chars: int = 60) -> bool:
    user = next(m["content"] for m in messages if m["role"] == "user")
    answer = normalize_answer(next(m["content"] for m in messages if m["role"] == "assistant"))
    return 0 < len(answer) <= max_chars and answer in normalize_answer(user)


def exact_match(prediction: str, gold: str) -> bool:
    return normalize_answer(prediction) == normalize_answer(gold)

In [ ]:
# explore: dữ liệu: tập train, tập val cho loss, tập trắc nghiệm cho exact-match (đều lấy từ val chia theo nhóm)
import json
import time

GPU = torch.cuda.is_available()
N_TRAIN, N_VAL, N_MC = (4000, 300, 200) if GPU else (32, 16, 16)
train_rows = [json.loads(line) for line in (CACHE / "m14-train.jsonl").open(encoding="utf-8")][:N_TRAIN]
val_rows = [json.loads(line) for line in (CACHE / "m14-val.jsonl").open(encoding="utf-8")]
rng = np.random.default_rng(0)
mc_pool = [r for r in val_rows if is_short_answer(r["messages"])]
mc_rows = [mc_pool[i] for i in sorted(rng.choice(len(mc_pool), size=N_MC, replace=False))]
mc_ids = {r["id"] for r in mc_rows}
loss_pool = [r for r in val_rows if r["id"] not in mc_ids]
loss_rows = [loss_pool[i] for i in sorted(rng.choice(len(loss_pool), size=N_VAL, replace=False))]
print(f"{'GPU' if GPU else 'CPU (bản thu nhỏ)'}: train {len(train_rows)}, val cho loss {len(loss_rows)}, "
      f"trắc nghiệm {len(mc_rows)} (trong {len(mc_pool)} câu đáp án ngắn của {len(val_rows)} câu val)")
print(mc_rows[0]["messages"][-2]["content"][-300:], "->", mc_rows[0]["messages"][-1]["content"])

In [ ]:
# explore: tokenize, batch theo ngân sách token, so tỉ lệ token thật với batch cố định 8 mẫu
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LEN, MAX_TOKENS = 512, 1024
tokenizer = AutoTokenizer.from_pretrained(MODEL)
train_ex = [e for e in (tokenize_chat(r["messages"], tokenizer, MAX_LEN) for r in train_rows) if e is not None]
loss_ex = [e for e in (tokenize_chat(r["messages"], tokenizer, MAX_LEN) for r in loss_rows) if e is not None]
lengths = [len(e["input_ids"]) for e in train_ex]
batches = token_budget_batches(lengths, MAX_TOKENS, seed=0)
real = sum(lengths)
padded_budget = sum(max(lengths[i] for i in b) * len(b) for b in batches)
padded_fixed = sum(max(lengths[s: s + 8]) * len(lengths[s: s + 8]) for s in range(0, len(lengths), 8))
labeled = sum(sum(label != IGNORE_INDEX for label in e["labels"]) for e in train_ex)
print(f"{len(train_ex)} mẫu, {real} token, {labeled} token có label ({labeled / real:.0%}); {len(batches)} batch")
print(f"tỉ lệ token thật: ngân sách token {real / padded_budget:.3f} | 8 mẫu theo thứ tự gốc {real / padded_fixed:.3f}")
eval_batches = [pad_batch([loss_ex[i] for i in b], tokenizer.pad_token_id)
                for b in token_budget_batches([len(e["input_ids"]) for e in loss_ex], MAX_TOKENS, seed=0)]

In [ ]:
# explore: model gốc: loss validation và exact-match trắc nghiệm
DEVICE = "cuda" if GPU else "cpu"
DTYPE = torch.bfloat16 if GPU else torch.float32


@torch.no_grad()
def answer_mc(model, rows, batch_size=32, max_new_tokens=32):
    tokenizer.padding_side = "left"
    preds = []
    for s in range(0, len(rows), batch_size):
        prompts = [tokenizer.apply_chat_template(r["messages"][:-1], tokenize=False, add_generation_prompt=True)
                   for r in rows[s: s + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tokenizer.pad_token_id)
        preds += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    return preds


def evaluate(model, name):
    t0 = time.perf_counter()
    nll, n_tok = heldout_nll(model, eval_batches)
    preds = answer_mc(model, mc_rows)
    golds = [r["messages"][-1]["content"] for r in mc_rows]
    em = float(np.mean([exact_match(p, g) for p, g in zip(preds, golds)]))
    contains = float(np.mean([normalize_answer(g) in normalize_answer(p) for p, g in zip(preds, golds)]))
    res = {"val_nll": round(nll, 4), "val_ppl": round(math.exp(nll), 3), "val_tokens": n_tok, "mc_em": em,
           "mc_contains": contains,
           "mc_mean_chars": float(np.mean([len(p) for p in preds])), "seconds": round(time.perf_counter() - t0, 1),
           "examples": preds[:5]}
    print(name, json.dumps(res, ensure_ascii=False))
    return res


torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE)
results = {"device": torch.cuda.get_device_name() if GPU else "cpu", "base": evaluate(model, "gốc")}

In [ ]:
# explore: SFT: LoRA của PEFT trên mọi lớp tuyến tính, AdamW, warmup rồi cosine, 1 epoch, gradient checkpointing
from peft import LoraConfig, get_peft_model
from transformers import get_cosine_schedule_with_warmup

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                                         task_type="CAUSAL_LM"))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"tham số train được {trainable:,} / {total:,} ({trainable / total:.2%})")
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.0)
sched = get_cosine_schedule_with_warmup(opt, max(1, len(batches) // 33), len(batches))
if GPU:
    torch.cuda.reset_peak_memory_stats()
model.train()
history, t0 = [], time.perf_counter()
for step, b in enumerate(batches):
    batch = {k: v.to(DEVICE) for k, v in pad_batch([train_ex[i] for i in b], tokenizer.pad_token_id).items()}
    loss = model(**batch).loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    opt.zero_grad(set_to_none=True)
    history.append(loss.item())
    if step % 50 == 0 or step == len(batches) - 1:
        print(f"bước {step:4d}/{len(batches)}  loss {np.mean(history[-50:]):.3f}  {time.perf_counter() - t0:.0f} s")
results["train"] = {"steps": len(batches), "seconds": round(time.perf_counter() - t0, 1), "trainable": trainable,
                    "first50_loss": float(np.mean(history[:50])), "last50_loss": float(np.mean(history[-50:])),
                    "peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2) if GPU else None}
print(results["train"])
results["sft"] = evaluate(model, "SFT")

In [ ]:
# explore: so với model lớn hơn không fine-tune (Qwen2.5-3B-Instruct bf16 cần khoảng 7 GB: chỉ chạy khi GPU đủ)
del model, opt
if GPU:
    torch.cuda.empty_cache()
free_gib = torch.cuda.mem_get_info()[0] / 2**30 if GPU else 0
if GPU and free_gib > 8:
    big = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", dtype=DTYPE).to(DEVICE).eval()
    results["qwen3b"] = evaluate(big, "3B không fine-tune")
    del big
else:
    print(f"bỏ qua 3B: GPU trống {free_gib:.1f} GiB (cần khoảng 8 GiB)")
path = CACHE / ("m14-sft-instruction.json" if GPU else "m14-sft-instruction-cpu.json")
path.write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")
print("đã ghi", path)

In [ ]:
# explore: bảng tổng hợp (kernel CPU in thêm kết quả GPU đã cache nếu có)
import pandas as pd


def table(res):
    names = {"base": "0.5B gốc", "sft": "0.5B + SFT LoRA", "qwen3b": "3B gốc, không fine-tune"}
    return pd.DataFrame({names[k]: {"val loss": res[k]["val_nll"], "val ppl": res[k]["val_ppl"],
                                    "trắc nghiệm EM": res[k]["mc_em"], "có chứa đáp án": res[k]["mc_contains"],
                                    "độ dài trả lời TB (ký tự)": res[k]["mc_mean_chars"]}
                         for k in names if k in res}).T.round(3)


display(table(results))
gpu_cache = CACHE / "m14-sft-instruction.json"
if not GPU and gpu_cache.exists():
    print("Kết quả GPU đã cache:")
    display(table(json.loads(gpu_cache.read_text(encoding="utf-8"))))

## Đọc kết quả

- Batch theo ngân sách token tăng tỉ lệ token thật bao nhiêu so với batch cố định 8 mẫu? Bao nhiêu phần trăm token
  có label: model học trên phần nào của dữ liệu?
- Loss validation giảm bao nhiêu sau SFT? Perplexity giảm có nghĩa là model **giỏi hơn** hay chỉ **giống phong cách**
  dữ liệu hơn? Exact-match trắc nghiệm nói gì về câu hỏi đó?
- So 0.5B sau SFT với 3B không fine-tune ở cả exact-match lẫn "có chứa đáp án" (đáp án đúng nằm đâu đó trong câu trả lời):
  hai thước đo kể hai câu chuyện khác nhau thế nào? Với bài toán này, thứ bạn thiếu là hành vi hay năng lực của model?
- Đọc vài câu trả lời mẫu trước và sau: độ dài, định dạng thay đổi thế nào?

Sau đó làm phần trắc nghiệm của bài trong app.